# Compare Bond, Rittinger and Kick on one reduction

**Question:** Where do Bond, Rittinger and Kick agree, and how do their predictions diverge away from calibration?

Level: 3. Prerequisites: Levels 1 and 2 in this topic. Reading time: 24 minutes.



## Objectives

Compare three comminution hypotheses over the same feed-size and product-size range, identify the calibration point at which they agree, and state what the comparison can and cannot establish.

## Setup

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0 scikit-learn==1.5.2 ruff==0.15.18 pytest==9.1.1
```

The OreFlow pipeline is imported from `data-pipeline/`; this notebook never installs packages.

For the level 3 01-comminution lesson, record the pinned versions before analysis so the environment behind its tables and figures can be reconstructed.

In [ ]:
# Import analysis tools and record their exact versions.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path("../../../").resolve()
sys.path.insert(0, str(repo / "data-pipeline"))
for package in ("numpy", "scipy", "pandas", "matplotlib", "nbclient", "nbformat", "ipykernel"):
    print(package, importlib.metadata.version(package))
legend_style = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False)


## Data

Owner: OreFlow authors, MIT project licence. The production case catalog supplies the soft-porphyry ore, crusher, mill, classifier and nominal operating definition. Physical constants and the energy-law reference endpoints come from engine/data/constants.json. Circuit outputs and size distributions are calculated here, rather than loaded from a saved trace. No measured feed distribution, power campaign or new external dataset is introduced. Source: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Method

Bond: $W=W_i(10/\sqrt{P}-10/\sqrt{F})$. Rittinger: $E=K_R(1/P-1/F)$; Kick: $E=K_K\ln(F/P)$. OreFlow calibrates $K_R$ and $K_K$ so all three match Bond at the declared 10,000 µm to 150 µm reference reduction. Away from that point they encode different size-reduction hypotheses. `law_curves` is the authoritative implementation used here.

## Comparison

Bond's work index is a standard-circuit empirical size-reduction method formalized in the Global Mining Guidelines Group's Bond Efficiency guideline (GMG01-MP-2021). Rittinger and Kick provide surface-area and reduction-ratio hypotheses; none is universally valid across every size regime. OreFlow reports Bond and keeps the other laws as diagnostic comparisons. This is a comparison of equations on an authored case, not a claim of superior prediction.

The physical comparison uses freshly achieved mill-feed F80 throughout. Crusher-feed F80 belongs to an upstream reduction and is not substituted into this mill calculation. The separately declared reference reduction determines the diagnostic constants but does not redefine the operating circuit.

In [ ]:
# Resolve the physical mill endpoints independently of the calibration reduction.
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.engine.energy import law_curves, reference_constants
from pipeline.engine.constants import constant

case = CASE_BY_ID["copper_porphyry_soft"]
nominal = simulate(case.ore, case.plant, case.nominal)
metrics = nominal.metrics
wi = case.nominal.work_index_kwh_t
f80 = nominal.grinding.feed_f80_um
print("Achieved mill F80 (µm):", round(f80, 3))
print("Achieved product P80 (µm):", round(nominal.grinding.p80_um, 3))


All operating curves use the same achieved mill feed and work index. A second calculation evaluates all three laws exactly at the production reference feed and product sizes, and inserts that exact product into its own plotting grid. Equality there follows from calibration, whereas equality at 150 µm with a different feed is not imposed.

In [ ]:
# Evaluate operating curves and the exact declared calibration as separate experiments.
p80 = np.geomspace(50.0, f80 / 1.01, 80)
curves = law_curves(wi, f80, p80)
reference_feed = float(constant("bond.reference_feed_um"))
reference_product = float(constant("bond.reference_product_um"))
calibration = law_curves(wi, reference_feed, np.array([reference_product]))
reference_products = np.unique(
    np.r_[np.geomspace(50.0, reference_feed / 1.01, 80), reference_product]
)
reference_curves = law_curves(wi, reference_feed, reference_products)
display(pd.DataFrame(calibration).round(6))


The left panel evaluates the physical mill-feed reduction. The right panel is a different, explicitly labelled calibration experiment; each marker uses the exact law value at the declared reference endpoint. No nearest-grid ordinate is used for a marker at another size.

In [ ]:
# Separate physical mill-feed curves from the exact reference-calibration figure.
plt.rcParams.update({"font.size": 22, "axes.titlepad": 40})
fig, axes = plt.subplots(2, 1, figsize=(6.4, 4.8))
for name in ("bond", "rittinger", "kick"):
    axes[0].plot(curves["p80_um"], curves[name], label=name.title())
    axes[1].plot(reference_products, reference_curves[name], label=name.title())
    axes[1].scatter([reference_product], [calibration[name][0]], s=28)
axes[0].set(title=f"Physical mill feed: F80={f80:.1f} µm", ylabel="Specific energy\n(kWh/t)")
axes[1].set(
    title=f"Calibration: F80={reference_feed:g} µm; P80={reference_product:g} µm",
    ylabel="Specific energy\n(kWh/t)",
)
for axis in axes:
    axis.set_xscale("log")
    axis.set_xlabel("Product P80 (µm)")
    axis.invert_xaxis()
    axis.grid(alpha=0.25)
    axis.legend()
fig.subplots_adjust(left=0.24, right=0.96, bottom=0.15, top=0.91, hspace=1.0)


The left curves can differ at 150 µm because their feed is the achieved mill feed. The right markers coincide only at the declared reference reduction, evaluated exactly rather than interpolated from an unrelated grid point. These are two distinct questions with labelled endpoints. The later nominal-work table and ratio sweep retain the physical mill-feed basis; the calibration panel does not supply a second physical operating state or measured energy validation.

Quantify the spread at the actual case reduction. Interpret it as structural disagreement between hypotheses, not a confidence interval or observed plant error.

In [ ]:
# Compare the three implemented values at this case's recorded product P80.
actual_p80 = float(metrics["p80_um"])
point = law_curves(wi, f80, np.array([actual_p80]))
comparison = pd.DataFrame(
    {
        "law": ["Bond", "Rittinger", "Kick"],
        "energy_kwh_t": [point["bond"][0], point["rittinger"][0], point["kick"][0]],
    }
)
comparison["relative_to_bond"] = comparison.energy_kwh_t / comparison.energy_kwh_t.iloc[0]
display(comparison)


Resolve the nominal circuit before evaluating alternative energy hypotheses. The achieved mill feed and product sizes, rather than the requested target alone, define a physical reduction. Bond, Rittinger and Kick all use the same work index and endpoints in this experiment. Their reference calibration agrees at a specified reduction, but that agreement is imposed and does not validate their response elsewhere.

In [ ]:
# Resolve the nominal circuit before evaluating alternative energy hypotheses.
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.engine.energy import law_curves

case = CASE_BY_ID["copper_porphyry_soft"]
nominal = simulate(case.ore, case.plant, case.nominal)
f80 = nominal.grinding.feed_f80_um
wi = case.nominal.work_index_kwh_t
print("Achieved mill F80 (µm):", round(f80, 2))


Calculate the calibrated work at the achieved product size. These are three alternative requirements, not energy components that should be added. Comparing them to net grinding energy requires a consistent circuit and efficiency basis. Keep the returned values in kWh per tonne of fresh feed so a downstream power comparison does not accidentally use circulating mill tonnage.

In [ ]:
# Calculate the calibrated work at the achieved product size.
at_product = law_curves(wi, f80, np.array([nominal.grinding.p80_um]))
display(pd.DataFrame(at_product).round(3))


Ask the engine to achieve a finer product while holding the fresh feed and ore fixed. This isolates liberation demand from throughput. A power-limited mill may fail to reach the requested target, which is why both requested and achieved sizes belong in the table. A finer target is a controlled model perturbation, not an operating recommendation for this ore.

In [ ]:
# Ask the engine to achieve a finer product while holding the fresh feed and ore fixed.
fine_point = case.nominal.with_values(target_p80_um=case.nominal.target_p80_um * 0.8)
fine = simulate(case.ore, case.plant, fine_point)
print("Requested product P80 (µm):", fine_point.target_p80_um)
print("Achieved product P80 (µm):", round(fine.grinding.p80_um, 3))
print("Required power (kW):", round(fine.grinding.required_power_kw, 3))


Repeat with a coarser target to bracket the nominal operating point. Coarsening can lower grinding demand while leaving more valuable mineral locked. The full circuit computes that downstream consequence rather than assigning recovery from an energy curve. A ranking by energy alone would miss the loss of recovered metal, so retain recovery and power together for the final comparison.

In [ ]:
# Repeat with a coarser target to bracket the nominal operating point.
coarse_point = case.nominal.with_values(target_p80_um=case.nominal.target_p80_um * 1.2)
coarse = simulate(case.ore, case.plant, coarse_point)
print("Coarse recovery (%):", round(coarse.metrics["recovery_pct"], 3))
print("Coarse required power (kW):", round(coarse.grinding.required_power_kw, 3))


Evaluate the three laws along the same logarithmic product-size sweep. The feed endpoint is fixed and every requested product is smaller than it. Normalize each alternative by Bond only after calculating its energy, since ratios hide the absolute demand. A crossing is agreement of calibrated equations at one size, not a physical phase transition or evidence that their assumptions become interchangeable.

In [ ]:
# Evaluate the three laws along the same logarithmic product-size sweep.
products = np.geomspace(max(50.0, f80 / 30), f80 / 1.1, 40)
sweep = law_curves(wi, f80, products)
ratio = {
    name: np.asarray(sweep[name]) / np.asarray(sweep["bond"]) for name in ("rittinger", "kick")
}
print("Maximum Rittinger / Bond:", round(float(max(ratio["rittinger"])), 3))


Plot relative energy-law divergence to distinguish calibration agreement from the shape of each reduction hypothesis.

In [ ]:
# Plot relative energy-law divergence to distinguish calibration agreement from the shape of each
# reduction hypothesis..
plt.rcParams.update({"font.size": 18, "axes.titlepad": 64})
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for name, values in ratio.items():
    ax.plot(products, values, label=name.title())
ax.axhline(1.0, color="black", linestyle="--", label="Bond")
ax.set(
    xscale="log",
    xlabel="Product P80 (µm)",
    ylabel="Energy / Bond\n(1)",
    title="Mill-feed energy-law ratios",
)
ax.legend(ncol=3, **legend_style)
fig.subplots_adjust(left=0.22, right=0.96, bottom=0.23, top=0.61)
ax.locator_params(axis="x", numticks=4)
ax.tick_params(axis="x", which="minor", labelbottom=False)


The ratio curves identify where a surface-area or reduction-ratio hypothesis predicts a different demand from Bond. Unity is a comparison baseline, not a confidence interval. The sweep deliberately holds feed and work index fixed, so it cannot explain geological hardness changes. Use the solved fine and coarse circuits to judge whether the desired product is reachable under installed power before treating an equation's energy requirement as an achievable process state.

A calibration equality needs two declared endpoints, not just a work index. OreFlow determines Rittinger and Kick constants from the same reference reduction used for Bond, then keeps those constants fixed for the case sweep. Changing the reference would change their numerical disagreement without changing the mathematical hypotheses. When reproducing this comparison with an ore test, record feed and product size conventions, the energy basis and the calibration reduction together. Agreement at a point selected for calibration supplies no additional independent observation.

## Results

**Answer:** For the same nominal feed/product pair, Bond predicts <!-- result:Bond work (kWh/t) -->0<!-- /result --> kWh/t, Rittinger <!-- result:Rittinger work (kWh/t) -->0<!-- /result --> and Kick <!-- result:Kick work (kWh/t) -->0<!-- /result -->. The common reference equality is imposed by calibration; the size-sweep figure shows the hypotheses diverging elsewhere. That disagreement is an equation comparison on authored ore, without measured-energy validation.

Bind the physical mill-endpoint law values separately from the exact reference-calibration table above.

In [ ]:
# Summarize the computed outcomes used in this lesson's interpretation.
answer_values = {
    "Nominal P80 (µm)": nominal.grinding.p80_um,
    "Requested fine P80 (µm)": fine_point.target_p80_um,
    "Achieved fine P80 (µm)": fine.grinding.p80_um,
    "Fine required power (kW)": fine.grinding.required_power_kw,
    "Bond work (kWh/t)": at_product["bond"][0],
    "Rittinger work (kWh/t)": at_product["rittinger"][0],
    "Kick work (kWh/t)": at_product["kick"][0],
}
answer_values = {label: round(float(number), 3) for label, number in answer_values.items()}
print(json.dumps(answer_values, ensure_ascii=False))


## Summary

A controlled same-case comparison makes the model choice visible: calibration makes laws agree at one point, not everywhere. Bond's stated fineness qualification and its energy basis still apply.

## References

- https://www.globalminingguidelines.org/guidelines/bond-efficiency/
- https://doi.org/10.1016/0301-7516(80)90034-4
- https://doi.org/10.1016/0301-7516(80)90034-4